In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
import pyquist as pq
import icm_plotly
from icm_plotly import RED, GOLD, STEEL, TEAL

Drag the sliders: the red curve is $x(t) = a \sin(2 \pi f t + \phi)$,
redrawn live. The gray curve keeps the starting parameters for reference,
and the dashed teal lines mark one period $t_0 = 1/f$.

In [ ]:
# hide
# autorun
T_MS = 20.0                             # window: 20 ms of signal
t = np.linspace(0.0, T_MS / 1000, 900)

A0, F0, PHI0 = 0.8, 220.0, 0.0          # the fixed reference parameters
y_ref = A0 * np.sin(2 * np.pi * F0 * t + PHI0)

def period_marks(f, phi):
    # one period, bracketed from the first rising zero crossing
    start = ((-phi / (2 * np.pi)) % 1.0) / f
    return start, start + 1 / f

def figure():
    fig = go.Figure()
    fig.add_scatter(x=t * 1000, y=y_ref, mode="lines",
                    line=dict(color=STEEL, width=1.6))
    fig.add_scatter(x=t * 1000, y=y_ref, mode="lines",
                    line=dict(color=RED, width=2.2))
    for y in (A0, -A0):
        fig.add_scatter(x=[0, T_MS], y=[y, y], mode="lines",
                        line=dict(color=GOLD, width=1.2, dash="dash"))
    p0, p1 = period_marks(F0, PHI0)
    for x in (p0, p1):
        fig.add_scatter(x=[x * 1000, x * 1000], y=[-1.05, 1.05], mode="lines",
                        line=dict(color=TEAL, width=1.2, dash="dash"))
    fig.update_xaxes(range=[0, T_MS], title_text="Time (ms)", fixedrange=True)
    fig.update_yaxes(range=[-1.05, 1.05], title_text="Amplitude",
                     fixedrange=True)
    return fig

def controls(fig):
    amp = widgets.FloatSlider(description="Amplitude a", min=0, max=1,
                              value=A0, step=0.01)
    freq = widgets.FloatSlider(description="Frequency f (Hz)", min=110,
                               max=880, value=F0, step=5)
    phase = widgets.FloatSlider(description="Phase φ (rad)", min=0,
                                max=round(2 * np.pi, 2), value=PHI0, step=0.05)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(a, f, phi, t=t, readout=readout):
        p0, p1 = period_marks(f, phi)
        with fig.batch_update():
            fig.data[1].y = a * np.sin(2 * np.pi * f * t + phi)
            fig.data[2].y = [a, a]
            fig.data[3].y = [-a, -a]
            fig.data[4].x = [p0 * 1000, p0 * 1000]
            fig.data[5].x = [p1 * 1000, p1 * 1000]
        readout.value = (
            f"<span style='font-size:0.9em'>period t₀ = 1/f = "
            f"{1000 / f:.2f} ms &nbsp;·&nbsp; angular frequency "
            f"ω = 2πf ≈ {2 * np.pi * f:.0f} rad/s</span>"
        )

    widgets.interactive_output(update, {"a": amp, "f": freq, "phi": phase})
    return widgets.VBox([amp, freq, phase, readout])

icm_plotly.show(figure, controls)

**Now hear it.** Edit the three parameters in the cell below and re-run
it: $f$ changes the pitch, $a$ changes the loudness, and $\phi$ changes
the picture but not the sound.

In [ ]:
a = 0.5      # amplitude: try 0.05 or 0.005 (keep it below 1!)
f = 220.0    # frequency in Hz: try 330 or 440
phi = 0.0    # initial phase in radians: try np.pi / 2 or np.pi

sr = 44100
tt = np.arange(sr) / sr                    # one second of sample times
x = a * np.sin(2 * np.pi * f * tt + phi)
x[:441] *= np.linspace(0, 1, 441)          # 10 ms fades: no clicks
x[-441:] *= np.linspace(1, 0, 441)
pq.play(pq.Audio(x.astype(np.float32), sr))